# 🦺 PPE Detection System — YOLOv8n Training Notebook
**Antigravity Edge Compute Platform | Computer Vision Engineer Assignment**

---

| Step | Description |
|------|-------------|
| 1 | Verify GPU & install dependencies |
| 2 | Upload dataset zip & configure data.yaml |
| 3 | Train YOLOv8n for 50 epochs on T4 GPU |
| 4 | Download trained `best.pt` weights |

> ⚡ **Enable GPU before running:** `Runtime → Change runtime type → T4 GPU`

---
## 🔧 Step 1 — Environment Setup & GPU Verification

In [ ]:
import sys
import torch

# Verify GPU
!nvidia-smi

# Install Ultralytics (includes YOLOv8)
!pip install -q ultralytics

print(f'\nPython  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'\n✅ GPU Ready: {torch.cuda.is_available()}')

---
## 📂 Step 2 — Upload Dataset & Configure data.yaml

**Instructions:**
1. On your Windows machine, right-click your `dataset` folder → **Send to → Compressed (zipped) folder**
2. Run the cell below — a file picker will appear
3. Select your `dataset.zip`

In [ ]:
import os
from google.colab import files

# --- Upload dataset zip ---
print('📂 Select your dataset.zip file...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {zip_name}')

# --- Extract ---
!mkdir -p /content/dataset
!unzip -q "{zip_name}" -d /content/dataset/

print('\n📁 Extracted contents:')
!find /content/dataset -maxdepth 3 | head -20

In [ ]:
import os
import yaml

BASE = '/content/dataset'

def find_split(base, split_name):
    """Auto-detect image directory for a given split."""
    candidates = [
        f'{base}/{split_name}/images',
        f'{base}/dataset/{split_name}/images',
        f'{base}/{split_name}',
    ]
    for c in candidates:
        if os.path.isdir(c):
            imgs = [f for f in os.listdir(c) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if imgs:
                return c, len(imgs)
    return None, 0

# --- Detect splits ---
print('🔍 Detecting dataset splits...')
train_path, train_n = find_split(BASE, 'train')
valid_path, valid_n = find_split(BASE, 'valid')
test_path,  test_n  = find_split(BASE, 'test')

print(f'  train : {train_path}  ({train_n} images)')
print(f'  valid : {valid_path}  ({valid_n} images)')
print(f'  test  : {test_path}   ({test_n} images)')

if not train_path or not valid_path:
    raise RuntimeError('❌ Could not find train or valid splits — check your zip structure.')

# --- Write data.yaml with absolute Colab paths ---
YAML_PATH = '/content/data.yaml'
data_config = {
    'train': train_path,
    'val'  : valid_path,
    'test' : test_path if test_path else valid_path,
    'nc'   : 4,
    'names': ['Gloves', 'Vest', 'helmet', 'person']
}
with open(YAML_PATH, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f'\n✅ data.yaml written to: {YAML_PATH}')
!cat /content/data.yaml

---
## 🏋️ Step 3 — Train YOLOv8n (50 Epochs, T4 GPU)

Expected time: **~10–20 minutes** on Colab T4 GPU

In [ ]:
import time
from ultralytics import YOLO

PROJECT_DIR = '/content/runs'
RUN_NAME    = 'ppe_yolov8n_training'
YAML_PATH   = '/content/data.yaml'

print('=' * 60)
print('  PPE DETECTION — YOLOv8n TRAINING | Antigravity Platform')
print('=' * 60)
print(f'  Model   : yolov8n.pt  (pretrained baseline)')
print(f'  Dataset : {YAML_PATH}')
print(f'  Epochs  : 50')
print(f'  ImgSize : 640 x 640')
print(f'  Batch   : -1  (AutoBatch)')
print(f'  Device  : GPU (cuda:0)')
print('=' * 60 + '\n')

# Load pretrained YOLOv8n baseline
model = YOLO('yolov8n.pt')

t0 = time.time()

results = model.train(
    data     = YAML_PATH,
    imgsz    = 640,
    epochs   = 50,
    batch    = -1,       # AutoBatch — auto-estimates optimal batch for GPU VRAM
    device   = 0,        # cuda:0 (T4 GPU)
    project  = PROJECT_DIR,
    name     = RUN_NAME,
    exist_ok = True,
    patience = 10,       # Early stopping
    plots    = True,     # Training curves + confusion matrix
    save     = True,
    workers  = 2,
    verbose  = True,
)

elapsed = time.time() - t0
BEST_PT = f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt'

print(f'\n✅ Training complete in {elapsed/60:.1f} minutes')
print(f'   Best weights saved at: {BEST_PT}')

---
## 💾 Step 4 — Download Trained Weights

This downloads `best.pt` directly to your local machine.

In [ ]:
from google.colab import files

BEST_PT = '/content/runs/ppe_yolov8n_training/weights/best.pt'

print('📥 Downloading best.pt ...')
files.download(BEST_PT)
print('✅ best.pt downloaded successfully')
print('\nNext step: Run export_onnx.py and convert_fp16.py on your local machine.')